In [14]:
import pandas as pd
import numpy as np

In [28]:
def calculate_ir_metrics(test_dataset_path, agent_results_path):
    
    test_df = pd.read_csv(test_dataset_path) 
    agent_df = pd.read_csv(agent_results_path) 

    merged_df = pd.merge(test_df, agent_df, on='input', how='inner')
    
    mrr_scores = []
    ndcg_scores = []
    hits_at_1 = 0
    hits_at_3 = 0
    total = len(merged_df)

    print(f"Analyzing {total} samples...")

    for idx, row in merged_df.iterrows():
        # Clean the ground truth
        truth = str(row['context']).strip().lower()
        
        
        retrieved = [c.strip().lower() for c in str(row['contexts']).split('|||')]
        
        rank = 0
        for i, doc in enumerate(retrieved):
            if truth[:100] in doc or doc[:100] in truth:
                rank = i + 1
                break
        
        # --- MRR & Recall Calculation ---
        if rank == 1:
            mrr_scores.append(1.0)
            hits_at_1 += 1
            hits_at_3 += 1
            # NDCG Calculation: Gain is 1 at Rank 1. DCG = 1/log2(1+1) = 1.0
            ndcg_scores.append(1.0) 
        elif rank == 2:
            mrr_scores.append(0.5)
            hits_at_3 += 1
            # NDCG Calculation: Gain is 1 at Rank 2. DCG = 1/log2(2+1) = 0.6309
            ndcg_scores.append(1 / np.log2(2 + 1))
        elif rank == 3:
            mrr_scores.append(0.3333)
            hits_at_3 += 1
            # NDCG Calculation: Gain is 1 at Rank 3. DCG = 1/log2(3+1) = 0.5
            ndcg_scores.append(1 / np.log2(3 + 1))
        else:
            mrr_scores.append(0.0)
            ndcg_scores.append(0.0)

    # 3. Final Calculations
    metrics = {
        "MRR": sum(mrr_scores) / total,
        "NDCG@3": sum(ndcg_scores) / total,
        "Recall@1 (Hit Rate)": hits_at_1 / total,
        "Recall@3": hits_at_3 / total
    }
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")
    
    
    return metrics

In [29]:

test_path = r"C:\Uni\4th Year\GP\ECHO\data\chatbot\outputs\echo_agent_evaluation\evaluation_data\shrunk_dataset_132.csv"

reranker_agent_responses = r"C:\Uni\4th Year\GP\ECHO\data\chatbot\outputs\echo_agent_evaluation\new_agent_responses\full_agent_response.csv"

wo_reranker_agent_responses=r"C:\Uni\4th Year\GP\ECHO\data\chatbot\outputs\echo_agent_evaluation\agent_responses_wo_reranker\full_agent_response.csv"

In [30]:
#With Reranker
stats = calculate_ir_metrics(test_path, reranker_agent_responses)

Analyzing 132 samples...
MRR: 0.9268
NDCG@3: 0.9419
Recall@1 (Hit Rate): 0.8712
Recall@3: 0.9848


In [31]:
#Without Reranker
stats = calculate_ir_metrics(test_path, wo_reranker_agent_responses)

Analyzing 132 samples...
MRR: 0.8106
NDCG@3: 0.8438
Recall@1 (Hit Rate): 0.7045
Recall@3: 0.9394
